In [8]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2
from sklearn.impute import SimpleImputer
import joblib

train = pd.read_csv('ml_train.csv')
val = pd.read_csv('ml_val.csv')
test = pd.read_csv('ml_test.csv')

for df in [train, val, test]:
    df['order_purchase'] = pd.to_datetime(df['order_purchase'])
    df['order_estimated_delivery'] = pd.to_datetime(df['order_estimated_delivery'])

In [9]:
################################ ADD 4 features purchase_month and estimated_days until product delivery
def add_date_features(df):
    df['purchase_month'] = df['order_purchase'].dt.month
    df['estimated_days'] = (df['order_estimated_delivery'] - df['order_purchase']).dt.days
    return df

train = add_date_features(train)
val = add_date_features(val)
test = add_date_features(test)

In [10]:
########################   Computes the customer–seller distance in KM using the Haversine formula

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def add_distance(df):
    df['distance_km'] = haversine(df['customer_lat'], df['customer_lng'],
                                   df['avg_seller_lat'], df['avg_seller_lng'])
    return df

train = add_distance(train)
val = add_distance(val)
test = add_distance(test)

In [11]:
######################################## make a missing flag field for each column has missing values.
missing_geo_cols = ['avg_seller_lat', 'avg_seller_lng', 'distance_km']
missing_dim_cols = ['max_product_weight_grams','max_product_length_cm',
                     'max_product_height_cm','max_product_width_cm']
missing_text_cols = ['avg_product_name_length','avg_product_description_length',
                      'avg_product_photos_qty']
missing_payment_cols = ['num_payments','num_installments','total_payment_value',
                         'pay_boleto','pay_credit_card','pay_debit_card',
                         'pay_not_defined','pay_voucher']
missing_customer_geo_cols = ['customer_lat','customer_lng']

all_missing_cols = (missing_geo_cols + missing_dim_cols + missing_text_cols +
                     missing_payment_cols + missing_customer_geo_cols)

for col in all_missing_cols:
    train[f'{col}_missing'] = train[col].isna().astype(int)
    val[f'{col}_missing'] = val[col].isna().astype(int)
    test[f'{col}_missing'] = test[col].isna().astype(int)

C:\Users\acer\AppData\Local\Temp\ipykernel_8484\2048322658.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train[f'{col}_missing'] = train[col].isna().astype(int)
C:\Users\acer\AppData\Local\Temp\ipykernel_8484\2048322658.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  val[f'{col}_missing'] = val[col].isna().astype(int)
C:\Users\acer\AppData\Local\Temp\ipykernel_8484\2048322658.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor per

In [12]:
########################################### fill all missing values with median 
imputer = SimpleImputer(strategy='median')
imputer.fit(train[all_missing_cols])

train[all_missing_cols] = imputer.transform(train[all_missing_cols])
val[all_missing_cols] = imputer.transform(val[all_missing_cols])
test[all_missing_cols] = imputer.transform(test[all_missing_cols])

In [13]:
drop_cols = ['order_purchase', 'order_estimated_delivery']
train_final = train.drop(columns=drop_cols)
val_final = val.drop(columns=drop_cols)
test_final = test.drop(columns=drop_cols)

In [14]:
train_final.to_csv('ml_train_features.csv', index=False)
val_final.to_csv('ml_val_features.csv', index=False)
test_final.to_csv('ml_test_features.csv', index=False)

joblib.dump(imputer, 'imputer.pkl')

feature_list = [c for c in train_final.columns if c != 'on_time'] ## feature list
with open('feature_list.txt', 'w') as f:
    f.write('\n'.join(feature_list))